# Russell (Nicaragua) α-β — Chain separability (sweep workflow)

Same workflow as the final CD4/CD8 and Rosati chain analyses, label = **chain (alpha vs beta)**.
Sanity check: expect ~1.0 (different loci). Single version (Russell is pre-filtered to UMI≥5,
**uniform weights** — no abundance available).

Full plot set: main sweep, by descriptor, full-depth vs rarefied, depth-only baseline,
V-gene baseline, mean vs covariance.

## 1 — Paths + load descriptors

In [ ]:
import sys, glob, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPERTOIRE_DIR = '/home/immunologylab/bioinformatics/analysis/tcr_repertoire/scripts/repertoire'
CLOUDS_DIR     = '/home/immunologylab/bioinformatics/analysis/data/processed/russell/clouds_embedded'
LANDMARKS      = '/home/immunologylab/bioinformatics/analysis/data/processed/descriptors/landmarks_beta.npz'
sys.path.insert(0, REPERTOIRE_DIR)
import rep_descriptors as rd, rep_metrics as rm

EMB = [f'e{i}' for i in range(128)]

def parse_stem(fname):
    base = os.path.basename(fname).replace('.par.parquet','').replace('.parquet','')
    donor = '_'.join(base.split('_')[:-1])           # 'nica_10270'
    chain = 'alpha' if base.split('_')[-1]=='A' else 'beta'
    return donor, chain

files = sorted(glob.glob(f'{CLOUDS_DIR}/*.parquet'))
V, chain_lab, donor_lab = [], [], []
for f in files:
    df = pd.read_parquet(f, columns=EMB + ['w_log'])
    Z = df[EMB].to_numpy(np.float32); Z /= (np.linalg.norm(Z,axis=1,keepdims=True)+1e-8)
    w = df['w_log'].to_numpy(np.float32)
    V.append(rd.mean_cov_weighted_np(Z, w))
    donor, chain = parse_stem(f)
    chain_lab.append(chain); donor_lab.append(donor)
V = np.vstack(V)
chain_lab = np.array(chain_lab); donor_lab = np.array(donor_lab)
print('clouds:', V.shape[0], '| descriptor dim:', V.shape[1])
print('chains:', pd.Series(chain_lab).value_counts().to_dict())
print('donors with both chains:', (pd.Series(donor_lab).value_counts()==2).sum())

## 2 — Main sweep (label = chain)

In [ ]:
ref_sizes = [1, 2, 5, 10, 15, 20, 25]
sweep = rm.ref_size_sweep_auroc(V, chain_lab, ref_sizes, n_draws=100, seed=0)
ys = [sweep[r] for r in ref_sizes]

fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.plot(ref_sizes, ys, marker='o', markersize=9, linewidth=2.5, color='#065A82',
        markerfacecolor='#065A82', markeredgecolor='white', markeredgewidth=1.5, label='mean+cov')
for x, y in zip(ref_sizes, ys):
    ax.annotate(f'{y:.3f}', (x, y), textcoords='offset points', xytext=(0, 12), ha='center', fontsize=9, color='#21295C', fontweight='bold')
ax.axhline(0.5, color='gray', ls='--', lw=1, alpha=0.6)
ax.text(ref_sizes[-1], 0.515, 'chance (0.5)', ha='right', fontsize=9, color='gray')
ax.set_xlabel('Reference-set size  (number of reference clouds)', fontsize=12, fontweight='bold')
ax.set_ylabel('Mean ROC-AUC  (alpha vs beta)', fontsize=12, fontweight='bold')
ax.set_title('Russell: chain separability (alpha vs beta) vs reference-set size', fontsize=12)
ax.set_xticks(ref_sizes); ax.set_ylim(0.45, 1.02); ax.grid(True, alpha=0.25)
ax.legend(loc='lower right', fontsize=10)
plt.tight_layout(); plt.show()
print('ref_size : mean ROC-AUC')
for r in ref_sizes: print(f'   {r:>2}    :  {sweep[r]:.4f}')

## 3 — Sweep by descriptor (mean+cov vs occupancy)

In [ ]:
curves = {'mean+cov': ys}; Vo = None
if os.path.exists(LANDMARKS):
    protos = np.load(LANDMARKS)['centroids']
    Vo = []
    for f in files:
        df = pd.read_parquet(f, columns=EMB + ['w_log'])
        Z = df[EMB].to_numpy(np.float32); Z /= (np.linalg.norm(Z,axis=1,keepdims=True)+1e-8)
        w = df['w_log'].to_numpy(np.float32)
        Vo.append(rd.weighted_occupancy(Z, w, protos, tau=0.1))
    Vo = np.vstack(Vo)
    sweep_o = rm.ref_size_sweep_auroc(Vo, chain_lab, ref_sizes, n_draws=100, seed=0)
    curves['occupancy'] = [sweep_o[r] for r in ref_sizes]
    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    colors = {'mean+cov':'#065A82', 'occupancy':'#C0392B'}
    for name, yv in curves.items():
        ax.plot(ref_sizes, yv, marker='o', markersize=8, linewidth=2.3, label=name, color=colors[name])
    ax.axhline(0.5, color='gray', ls='--', lw=1, alpha=0.6)
    ax.set_xlabel('Reference-set size  (number of reference clouds)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Mean ROC-AUC  (alpha vs beta)', fontsize=12, fontweight='bold')
    ax.set_title('Russell: chain sweep by descriptor', fontsize=12)
    ax.set_xticks(ref_sizes); ax.set_ylim(0.45, 1.02); ax.grid(True, alpha=0.25)
    ax.legend(loc='lower right', fontsize=10)
    plt.tight_layout(); plt.show()
    print('occupancy:', {r: round(sweep_o[r],4) for r in ref_sizes})
else:
    print('landmarks not found - mean+cov only')

## 4 — Full-depth vs rarefied

In [ ]:
sizes = [len(pd.read_parquet(f, columns=['w_log'])) for f in files]
K = int(np.percentile(sizes, 5))
print('rarefying to K =', K, '| min cloud =', min(sizes))

def descriptor_rarefied(f, K, seed):
    df = pd.read_parquet(f, columns=EMB + ['w_log'])
    if len(df) > K: df = df.sample(K, random_state=seed)
    Z = df[EMB].to_numpy(np.float32); Z /= (np.linalg.norm(Z,axis=1,keepdims=True)+1e-8)
    w = df['w_log'].to_numpy(np.float32); w = w / w.sum()
    return rd.mean_cov_weighted_np(Z, w)

rare = {r: [] for r in ref_sizes}
for seed in range(5):
    Vr = np.vstack([descriptor_rarefied(f, K, seed) for f in files])
    sw = rm.ref_size_sweep_auroc(Vr, chain_lab, ref_sizes, n_draws=100, seed=seed)
    for r in ref_sizes: rare[r].append(sw[r])
rare_mean = [np.mean(rare[r]) for r in ref_sizes]; rare_std = [np.std(rare[r]) for r in ref_sizes]

fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.plot(ref_sizes, ys, marker='o', markersize=8, linewidth=2.5, color='#065A82', label='full depth')
ax.errorbar(ref_sizes, rare_mean, yerr=rare_std, marker='s', markersize=7, linewidth=2.5, color='#C0392B', label=f'rarefied to K={K}', capsize=3)
for x, y in zip(ref_sizes, ys): ax.annotate(f'{y:.3f}',(x,y),textcoords='offset points',xytext=(0,12),ha='center',fontsize=8.5,color='#065A82',fontweight='bold')
for x, y in zip(ref_sizes, rare_mean): ax.annotate(f'{y:.3f}',(x,y),textcoords='offset points',xytext=(0,-16),ha='center',fontsize=8.5,color='#C0392B',fontweight='bold')
ax.axhline(0.5, color='gray', ls='--', lw=1, alpha=0.6)
ax.set_xlabel('Reference-set size  (number of reference clouds)', fontsize=12, fontweight='bold')
ax.set_ylabel('Mean ROC-AUC  (alpha vs beta)', fontsize=12, fontweight='bold')
ax.set_title('Russell: chain sweep, full depth vs rarefied', fontsize=12)
ax.set_xticks(ref_sizes); ax.set_ylim(0.45, 1.02); ax.grid(True, alpha=0.25)
ax.legend(loc='lower right', fontsize=10)
plt.tight_layout(); plt.show()

## 5 — Sweep vs depth-only baseline

In [ ]:
from sklearn.metrics import roc_auc_score
n_clono = np.array(sizes)
y_beta = (chain_lab == 'beta').astype(int)
depth_auc = roc_auc_score(y_beta, n_clono); depth_auc = max(depth_auc, 1 - depth_auc)
print(f'depth-only baseline: {depth_auc:.4f}')

fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.plot(ref_sizes, ys, marker='o', markersize=9, linewidth=2.5, color='#065A82', label='mean+cov (foundation)')
for x, y in zip(ref_sizes, ys): ax.annotate(f'{y:.3f}',(x,y),textcoords='offset points',xytext=(0,12),ha='center',fontsize=9,color='#21295C',fontweight='bold')
ax.axhline(depth_auc, color='#C0392B', ls='--', lw=1.8, alpha=0.8, label=f'depth-only baseline ({depth_auc:.3f})')
ax.axhline(0.5, color='gray', ls=':', lw=1, alpha=0.6)
ax.set_xlabel('Reference-set size  (number of reference clouds)', fontsize=12, fontweight='bold')
ax.set_ylabel('Mean ROC-AUC  (alpha vs beta)', fontsize=12, fontweight='bold')
ax.set_title('Russell: chain sweep vs depth-only baseline', fontsize=12)
ax.set_xticks(ref_sizes); ax.set_ylim(0.45, 1.02); ax.grid(True, alpha=0.25)
ax.legend(loc='lower right', fontsize=10)
plt.tight_layout(); plt.show()

## 6 — Sweep vs V-gene baseline

In [ ]:
vgene_data = []
for f in files:
    df = pd.read_parquet(f, columns=['v_gene','w_log'])
    vgene_data.append((df['v_gene'].to_numpy(), df['w_log'].to_numpy()))
vocab = rm.build_vgene_vocab([vg for vg,_ in vgene_data])
V_vg = np.vstack([rm.vusage_vector(vg, w, vocab) for vg,w in vgene_data])
sweep_vg = rm.ref_size_sweep_auroc(V_vg, chain_lab, ref_sizes, n_draws=100, seed=0)
ys_vg = [sweep_vg[r] for r in ref_sizes]

fig, ax = plt.subplots(figsize=(9, 5.8))
ax.plot(ref_sizes, ys, marker='o', markersize=8, linewidth=2.5, color='#065A82', label='mean+cov (foundation)')
if Vo is not None:
    ax.plot(ref_sizes, curves['occupancy'], marker='^', markersize=8, linewidth=2.3, color='#1C7293', label='occupancy')
ax.plot(ref_sizes, ys_vg, marker='s', markersize=8, linewidth=2.3, color='#C0392B', label='V-gene usage (baseline)')
ax.axhline(0.5, color='gray', ls='--', lw=1, alpha=0.6)
ax.set_xlabel('Reference-set size  (number of reference clouds)', fontsize=12, fontweight='bold')
ax.set_ylabel('Mean ROC-AUC  (alpha vs beta)', fontsize=12, fontweight='bold')
ax.set_title('Russell: chain sweep, foundation vs V-gene baseline', fontsize=12)
ax.set_xticks(ref_sizes); ax.set_ylim(0.45, 1.02); ax.grid(True, alpha=0.25)
ax.legend(loc='lower right', fontsize=10)
plt.tight_layout(); plt.show()
print('V-gene:', {r: round(sweep_vg[r],4) for r in ref_sizes})